# 04 — Customer Analysis

## 1. Objetivo del análisis

Hasta este punto, el análisis se ha centrado principalmente en el comportamiento de las **sesiones** dentro del e-commerce.

En este notebook cambiaremos la unidad de análisis:

> Pasaremos de estudiar qué ocurre dentro de una sesión a estudiar cómo se comportan los usuarios a través de múltiples sesiones.

El objetivo es comprender el comportamiento individual de los clientes y responder preguntas relacionadas con:

- recurrencia;
- frecuencia de interacción;
- conversión;
- comportamiento antes y después de una compra;
- comportamiento entre sesiones;
- recuperación de usuarios;
- intención de compra;
- relación entre diferentes sesiones de un mismo usuario.

Este análisis busca generar insights útiles para:

- **Marketing**
- **CRO (Conversion Rate Optimization)**
- **Commercial Analytics**
- **Customer Analytics**

No se busca todavía construir una segmentación RFM, analizar cohortes o calcular Customer Lifetime Value. Estos análisis se reservarán para notebooks posteriores.

---

# 2. Cambio de unidad de análisis

En el Notebook 03 la principal unidad de análisis fue la **sesión**.

En este notebook la unidad principal será el **usuario (`user_id`)**.

La estructura conceptual será:

```text
Evento
   ↓
Sesión
   ↓
Usuario

## **1. Carga y preparacion**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

In [2]:
import importlib
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "src" / "data" / "loading.py").exists():
    project_root = project_root.parent

if not (project_root / "src" / "data" / "loading.py").exists():
    raise FileNotFoundError("No se encontró la raíz del proyecto con src/data/loading.py")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

for module_name in ("src.data.loading", "src.data", "src"):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()

from src.data.loading import load_parquet

#### Definicion de paleta de colores para las visualizaciones

In [3]:
NS_BG = "#E8E8E8"
NS_INK = "#292929"
NS_ACCENT = "#1B6B45"

# Variantes de peso/alpha para jerarquía sin usar semántica roja/verde
NS_INK_70 = "#292929B3"   # alpha 70%
NS_INK_40 = "#29292966"   # alpha 40%
NS_INK_15 = "#29292926"   # alpha 15% — gridlines, bordes suaves

plt.rcParams.update({
    # Fuentes
    "font.family": "sans-serif",
    "font.sans-serif": ["Inter", "Arial", "DejaVu Sans"],
    "font.weight": "regular",

    # Colores base
    "figure.facecolor": NS_BG,
    "axes.facecolor": NS_BG,
    "savefig.facecolor": NS_BG,
    "text.color": NS_INK,
    "axes.labelcolor": NS_INK,
    "axes.titlecolor": NS_INK,
    "xtick.color": NS_INK_70,
    "ytick.color": NS_INK_70,

    # Ejes — sin caja, jerarquía por peso
    "axes.edgecolor": NS_INK_40,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": True,
    "axes.spines.bottom": True,

    # Grid — sutil, nunca protagonista
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": NS_INK_15,
    "grid.linewidth": 0.6,
    "grid.linestyle": "-",
    "axes.axisbelow": True,

    # Ciclo de color — monocromático con acento esmeralda como único highlight
    "axes.prop_cycle": plt.cycler(color=[NS_ACCENT, NS_INK, NS_INK_70, NS_INK_40]),

    # Tamaños
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,

    # Leyenda — sin caja, minimal
    "legend.frameon": False,
    "legend.loc": "best",

    # Export
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

#### Carga de datos

In [4]:
DATA_PATH = "../data/processed/"

In [5]:
df = load_parquet(DATA_PATH + "ecommerce_clean.parquet")

In [6]:
df.shape

(2074532, 10)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2074532 entries, 0 to 2074531
Data columns (total 10 columns):
 #   Column         Dtype              
---  ------         -----              
 0   event_time     datetime64[us, UTC]
 1   event_type     str                
 2   product_id     int64              
 3   category_id    int64              
 4   category_code  str                
 5   brand          str                
 6   price          float64            
 7   user_id        int64              
 8   user_session   str                
 9   month          period[M]          
dtypes: datetime64[us, UTC](1), float64(1), int64(3), period[M](1), str(4)
memory usage: 251.4 MB


In [8]:
df.head(5)

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,month
0,2019-10-01 00:01:46+00:00,view,5843665,1487580005092295511,NaN,f.o.x,9.44,462033176,a18e0999-61a1-4218-8f8f-61ec1d375361,2019-10
1,2019-10-01 00:01:55+00:00,cart,5868461,1487580013069861041,NaN,italwax,3.57,514753614,e2fecb2d-22d0-df2c-c661-15da44b3ccf1,2019-10
2,2019-10-01 00:02:50+00:00,view,5877456,1487580006300255120,NaN,jessnail,122.22,527418424,86e77869-afbc-4dff-9aa2-6b7dd8c90770,2019-10
3,2019-10-01 00:03:41+00:00,view,5649270,1487580013749338323,NaN,concept,6.19,555448072,b5f72ceb-0730-44de-a932-d16db62390df,2019-10
4,2019-10-01 00:03:44+00:00,view,18082,1487580005411062629,NaN,cnd,16.03,552006247,2d8f304b-de45-4e59-8f40-50c603843fe5,2019-10


## **2. Radiografia de clientes**

In [9]:
unique_users = df['user_id'].nunique()
unique_users

163781

In [10]:
# usuarios compradores y no compradores

buyers = df.loc[df["event_type"] == "purchase", "user_id"].nunique()

non_buyers = unique_users - buyers

user_conversion_rate = buyers / unique_users * 100

print(f"Usuarios totales: {unique_users}")
print(f"Compradores: {buyers}")
print(f"Usuarios que no compraron: {non_buyers}")
print(f"Proporción de conversión: {user_conversion_rate:.2f}%")

Usuarios totales: 163781
Compradores: 11040
Usuarios que no compraron: 152741
Proporción de conversión: 6.74%


In [11]:
# Usuarios single session vs recurrente

sessions_per_user = (
    df.groupby("user_id")["user_session"]
    .nunique()
)

# usuarios single-sesion
single_session_users = (sessions_per_user == 1).sum()

# usuarios recurrentes
repeat_users = (sessions_per_user > 1).sum()

# proporciones
single_session_rate = single_session_users / unique_users * 100
repeat_user_rate = repeat_users / unique_users * 100

print(f"Usuarios con una sola sesión: {single_session_users}")
print(f"Usuarios con 2+ sesiones: {repeat_users}")
print(f"Total de usuarios: {unique_users}")
print(f"Proporción de usuarios con una sola sesión: {single_session_rate:.2f}%")
print(f"Proporción de usuarios recurrentes: {repeat_user_rate:.2f}%")

Usuarios con una sola sesión: 108362
Usuarios con 2+ sesiones: 55403
Total de usuarios: 163781
Proporción de usuarios con una sola sesión: 66.16%
Proporción de usuarios recurrentes: 33.83%


In [12]:
sessions_per_user.describe()

count    163781.000000
mean          2.723527
std          11.202608
min           0.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        1713.000000
Name: user_session, dtype: float64

#### Nota sobre eventos sin `user_session`

Durante la construcción de la tabla `customer_summary` se identificaron **506 eventos (0.02% del total)** con `user_session` nulo, correspondientes a **146 usuarios (0.09% de la base)**.

Estos eventos corresponden principalmente a:

- `cart`: 417 eventos (82.41%)
- `remove_from_cart`: 85 eventos (16.80%)
- `view`: 4 eventos (0.79%)

No se identificaron eventos `purchase` sin `user_session`.

De los 146 usuarios afectados, **130 también presentan sesiones válidas**, mientras que **16 usuarios únicamente presentan eventos sin una sesión identificable**.

Debido a que no existe información suficiente para reconstruir de forma fiable estas sesiones, **no se realizó imputación de `user_session`**.

Los eventos se conservarán para análisis a nivel usuario, donde `user_id` permite atribuir correctamente la actividad al cliente. Sin embargo, estos eventos no se utilizarán para métricas que requieran identificar una sesión específica.

Para evitar interpretaciones incorrectas, la métrica de sesiones se denominará `known_sessions`, representando únicamente las sesiones identificables mediante `user_session`.

In [13]:
df[df["user_id"] == 559219168]

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,month
156754,2019-10-11 15:56:49+00:00,cart,5645768,1487580007717929935,NaN,NaN,1.59,559219168,NaN,2019-10


In [14]:
df["user_session"].isna().sum()

np.int64(506)

In [15]:
missing_sessions = df["user_session"].isna().sum()
total_events = len(df)

missing_session_rate = missing_sessions / total_events * 100

print(f"Eventos con user_session nulo: {missing_sessions:,}")
print(f"Porcentaje: {missing_session_rate:.2f}%")

Eventos con user_session nulo: 506
Porcentaje: 0.02%


In [16]:
users_with_missing_session = (
    df.loc[df["user_session"].isna(), "user_id"]
    .nunique()
)

print(f"Usuarios afectados: {users_with_missing_session:,}")

Usuarios afectados: 146


In [17]:
users_with_missing_session / df["user_id"].nunique() * 100

0.08914342933551511

In [18]:
df.loc[df["user_session"].isna(), "event_type"].value_counts()

event_type
cart                417
remove_from_cart     85
view                  4
Name: count, dtype: int64

In [19]:
df.loc[df["user_session"].isna(), "event_type"].value_counts(normalize=True) * 100

event_type
cart                82.411067
remove_from_cart    16.798419
view                 0.790514
Name: proportion, dtype: float64

In [20]:
users_missing_session = df.loc[
    df["user_session"].isna(), "user_id"
].unique()

user_session_check = (
    df[df["user_id"].isin(users_missing_session)]
    .groupby("user_id")["user_session"]
    .apply(lambda x: x.notna().any())
)

user_session_check.value_counts()

user_session
True     130
False     16
Name: count, dtype: int64

#### resumen

In [24]:
# resumen de los clientes

df = df.assign(
    is_cart=df["event_type"].eq("cart"),
    is_purchase=df["event_type"].eq("purchase"),
    purchase_revenue=np.where(df["event_type"].eq("purchase"), df["price"], 0),
    is_missing_session=df["user_session"].isna(),
)

customer_summary = (
    df.assign(event_date=df["event_time"].dt.normalize())
    .groupby("user_id", sort=False)
    .agg(
        known_sessions=("user_session", "nunique"),
        sessions=("user_session", "nunique"),
        events=("event_type", "size"),
        active_days=("event_date", "nunique"),
        products_interacted=("product_id", "nunique"),
        cart_events=("is_cart", "sum"),
        purchase_events=("is_purchase", "sum"),
        revenue=("purchase_revenue", "sum"),
        events_without_session=("is_missing_session", "sum"),
    )
    .reset_index()
)


In [25]:
customer_summary.head()

,user_id,known_sessions,sessions,events,active_days,products_interacted,cart_events,purchase_events,revenue,events_without_session
0,462033176,3,3,3,1,1,0,0,0.0,0
1,514753614,5,5,45,4,27,4,0,0.0,0
2,527418424,4,4,13,4,7,0,0,0.0,0
3,555448072,2,2,2,1,1,0,0,0.0,0
4,552006247,4,4,4,2,2,0,0,0.0,0


In [26]:
customer_summary.shape

(163781, 10)

In [27]:
customer_summary.info()

<class 'pandas.DataFrame'>
RangeIndex: 163781 entries, 0 to 163780
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   user_id                 163781 non-null  int64  
 1   known_sessions          163781 non-null  int64  
 2   sessions                163781 non-null  int64  
 3   events                  163781 non-null  int64  
 4   active_days             163781 non-null  int64  
 5   products_interacted     163781 non-null  int64  
 6   cart_events             163781 non-null  int64  
 7   purchase_events         163781 non-null  int64  
 8   revenue                 163781 non-null  float64
 9   events_without_session  163781 non-null  int64  
dtypes: float64(1), int64(9)
memory usage: 12.5 MB


In [28]:
customer_summary["user_id"].nunique() == len(customer_summary)

True

In [29]:
customer_summary.describe()

,user_id,known_sessions,sessions,events,active_days,products_interacted,cart_events,purchase_events,revenue,events_without_session
count,1.637810e+05,163781.000000,163781.000000,163781.000000,163781.000000,163781.000000,163781.000000,163781.000000,163781.000000,163781.000000
mean,5.579380e+08,2.723527,2.723527,12.666500,1.713404,6.216899,3.510566,0.778869,3.795004,0.003089
std,6.589670e+07,11.202608,11.202608,69.177258,2.781606,25.700202,21.097804,5.524093,25.485196,0.243168
min,4.661182e+06,0.000000,0.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,5.556370e+08,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,5.730579e+08,1.000000,1.000000,2.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,5.987132e+08,2.000000,2.000000,4.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000
max,6.220880e+08,1713.000000,1713.000000,8066.000000,116.000000,1635.000000,2676.000000,506.000000,1559.210000,78.000000


In [41]:
customer_summary[customer_summary["revenue"] > 0].shape

(11040, 10)

In [43]:
total_revenue_buyers = customer_summary.loc[
    customer_summary["revenue"] > 0, "revenue"
].sum()

print(f"Revenue total de usuarios compradores: {total_revenue_buyers:,.2f}")

Revenue total de usuarios compradores: 621,549.60


In [44]:
buyers_summary = customer_summary[
    customer_summary["purchase_events"] > 0
].copy()

In [45]:
top_10_buyers_n = int(len(buyers_summary) * 0.10)

print(f"Top 10% de compradores: {top_10_buyers_n:,}")

Top 10% de compradores: 1,104


In [46]:
buyers_sorted = buyers_summary.sort_values(
    "revenue",
    ascending=False
)

In [47]:
top_10_buyers = buyers_sorted.head(top_10_buyers_n)

In [48]:
top_10_buyer_revenue = top_10_buyers["revenue"].sum()
total_buyer_revenue = buyers_summary["revenue"].sum()

top_10_buyer_share = (
    top_10_buyer_revenue / total_buyer_revenue * 100
)

print(f"Revenue Top 10% de compradores: {top_10_buyer_revenue:,.2f}")
print(f"Revenue total de compradores: {total_buyer_revenue:,.2f}")
print(f"Participación: {top_10_buyer_share:.2f}%")

Revenue Top 10% de compradores: 263,768.62
Revenue total de compradores: 621,549.60
Participación: 42.44%


Al analizar únicamente a los usuarios que realizaron al menos una compra, se observa una concentración relevante del revenue.

El **10% de los compradores con mayor revenue**, equivalente a **1,104 compradores**, generó **Bs. 263,768.62**, representando el **42.44% del revenue total generado por compradores (Bs. 621,549.60)**.

Este resultado indica que una proporción relativamente pequeña de los compradores concentra una parte significativa de los ingresos del e-commerce.

> **Insight descriptivo:** existe una concentración relevante del revenue entre los compradores de mayor valor.

Este hallazgo no implica necesariamente que el negocio dependa de estos clientes ni que sean clientes recurrentes o fieles, ya que la clasificación se realizó retrospectivamente utilizando el revenue acumulado durante todo el período analizado. Se requeriría analizar recurrencia, frecuencia de compra y comportamiento temporal para determinar la naturaleza de estos compradores.

In [49]:
# concentracion del revenue

percentiles = [0.01, 0.05, 0.10, 0.20, 0.50, 1.00]

In [50]:
concentration = []

buyers_sorted = buyers_summary.sort_values(
    "revenue",
    ascending=False
)

total_revenue = buyers_sorted["revenue"].sum()

for p in percentiles:
    n_users = int(len(buyers_sorted) * p)

    top_users = buyers_sorted.head(n_users)

    revenue = top_users["revenue"].sum()

    revenue_share = revenue / total_revenue * 100

    concentration.append({
        "Top": f"{p:0%}",
        "Buyers": n_users,
        "Revenue": revenue,
        "Revenue Share": revenue_share,
    })


concentration_df = pd.DataFrame(concentration)

concentration_df

,Top,Buyers,Revenue,Revenue Share
0,1.000000%,110,65594.93,10.553451
1,5.000000%,552,181143.54,29.143859
2,10.000000%,1104,263768.62,42.437260
3,20.000000%,2208,366296.18,58.932735
4,50.000000%,5520,526168.79,84.654353
5,100.000000%,11040,621549.60,100.000000


### Insight — Concentración de Revenue

El revenue presenta una concentración significativa entre los compradores de mayor valor.

El **1% de los compradores concentra el 10.55% del revenue**, mientras que el **10% concentra el 42.44%**. A su vez, el **20% concentra el 58.93%** y el **50% de los compradores genera el 84.65% del revenue total**.

Estos resultados muestran que el valor económico no está distribuido uniformemente entre los compradores, sino que existe una estructura de concentración progresiva hacia los clientes de mayor valor.

Este análisis es descriptivo y retrospectivo: los compradores fueron ordenados según el revenue acumulado durante todo el período analizado. Por lo tanto, estos resultados no permiten afirmar todavía que los clientes de mayor valor sean más recurrentes, fieles o de mayor valor futuro.

Será necesario analizar posteriormente su frecuencia de compra, recurrencia y comportamiento para determinar qué características diferencian a estos compradores.

## **3. Comportamiento y recurrencia**

## **4. Customer journey entre sesiones**

## **5. Conversion y comportamiento por cliente**

## **6. Intencion y recuperacion**

## **7. Insights de customer analysis**

## **8. Conclusiones y oportunidades**